<a href="https://colab.research.google.com/github/Vulk123/defi-ai-agent/blob/main/1_train_agent_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pandas numpy scikit-learn tensorflow matplotlib


In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import MinMaxScaler


In [5]:
# Simulated dataset: each row = [tx_value, gas_used, tx_count, time_gap, ...]
np.random.seed(42)
data = np.random.rand(1000, 5) * 100

df = pd.DataFrame(data, columns=["value", "gas_used", "tx_count", "time_gap", "other"])
print(df.head())


       value   gas_used   tx_count   time_gap      other
0  37.454012  95.071431  73.199394  59.865848  15.601864
1  15.599452   5.808361  86.617615  60.111501  70.807258
2   2.058449  96.990985  83.244264  21.233911  18.182497
3  18.340451  30.424224  52.475643  43.194502  29.122914
4  61.185289  13.949386  29.214465  36.636184  45.606998


In [6]:
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(df)

print("Scaled shape:", X_scaled.shape)


Scaled shape: (1000, 5)


In [7]:
input_dim = X_scaled.shape[1]

# Encoder
input_layer = layers.Input(shape=(input_dim,))
encoded = layers.Dense(16, activation="relu")(input_layer)
encoded = layers.Dense(8, activation="relu")(encoded)

# Decoder
decoded = layers.Dense(16, activation="relu")(encoded)
decoded = layers.Dense(input_dim, activation="sigmoid")(decoded)

# Autoencoder
autoencoder = keras.Model(input_layer, decoded)
autoencoder.compile(optimizer="adam", loss="mse")

autoencoder.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 5)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │            96 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 5)              │            85 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 461 (1.80 KB)

 Trainable params: 461 (1.80 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
history = autoencoder.fit(
    X_scaled, X_scaled,
    epochs=30,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)


Epoch 1/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 0.0799 - val_loss: 0.0761
Epoch 2/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0766 - val_loss: 0.0713
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0707 - val_loss: 0.0656
Epoch 4/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0665 - val_loss: 0.0597
Epoch 5/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0592 - val_loss: 0.0542
Epoch 6/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0540 - val_loss: 0.0496
Epoch 7/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0485 - val_loss: 0.0448
Epoch 8/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0440 - val_loss: 0.0411
Epoch 9/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0393 - val_loss: 0.0371
Epoch 10/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0361 - val_loss: 0.0319
Epoch 11/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0307 - val_loss: 0.0270
Epoch 12/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0258 - val_l

In [9]:
# Reconstruction error
reconstructions = autoencoder.predict(X_scaled)
mse = np.mean(np.power(X_scaled - reconstructions, 2), axis=1)

threshold = np.percentile(mse, 95)  # top 5% as anomaly
print("Threshold:", threshold)


32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Threshold: 0.002331760236524138


In [10]:
import joblib

# Save autoencoder
autoencoder.save("models/autoencoder_model.keras")

# Save scaler
joblib.dump(scaler, "models/scaler.pkl")

# Save threshold
with open("models/threshold.txt", "w") as f:
    f.write(str(threshold))
